# 🏁 ASTRI INTERVIEW: OFFICIAL API SIMULATION (V5.0)

### System Architecture
- **Hybrid Engine**: N-gram (Statistical) + Bi-LSTM (Neural)
- **Interface**: Mocked official Trexsim API Response
- **Scope**: 30-word session with cumulative stats.

In [ ]:
import torch
import torch.nn as nn
import string, time, random, collections

class extract_tensor(nn.Module):
    def forward(self, x): return x[0][:, -1, :]

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.LSTM_stack = nn.Sequential(
            nn.Embedding(64, 32, max_norm=1, norm_type=2),
            nn.LSTM(input_size=32, hidden_size=64, num_layers=1, batch_first=True, bidirectional=True),
            extract_tensor(), nn.Linear(128, 26)
        )
    def forward(self, x): return self.LSTM_stack(x)

def encode_for_nn(word_str):
    clean = word_str.replace(' ', '')
    vector = [0] * 35
    start = 35 - len(clean)
    cmap = {'_': 27}
    for i, c in enumerate(string.ascii_lowercase): cmap[c] = i + 1
    for i, c in enumerate(clean):
        if start + i < 35: vector[start + i] = cmap.get(c, 27)
    return torch.tensor([vector], dtype=torch.long)

model = NeuralNetwork()
try:
    model.load_state_dict(torch.load('best_model_state.pt', map_location='cpu'))
    model.eval()
    print('✅ Engine A (Bi-LSTM): LOADED')
except:
    print('❌ Neural Weights missing.')

class NgramEngine:
    def __init__(self, dict_path):
        try:
            with open(dict_path, 'r') as f: self.full_dictionary = f.read().splitlines()
            self.letter_set = sorted(set(''.join(self.full_dictionary)))
            self.unigram, self.bigram = self.build_ngram(self.full_dictionary)
            print('✅ Engine B (Statistical): LOADED')
        except:
            print('❌ Dictionary not found.')
            self.letter_set = list(string.ascii_lowercase)
            self.unigram, self.bigram = {}, {}

    def build_ngram(self, dictionary):
        unigram = collections.defaultdict(lambda: collections.defaultdict(int))
        bigram = collections.defaultdict(lambda: collections.defaultdict(lambda: collections.defaultdict(int)))
        for word in dictionary:
            wl = len(word)
            for i in range(wl - 1): bigram[wl][word[i]][word[i+1]] += 1
            for letter in set(word): unigram[wl][letter] += 1
        return unigram, bigram

    def guess(self, word, guessed):
        probs = [0] * len(self.letter_set)
        for i, letter in enumerate(self.letter_set):
            if letter in guessed: continue
            try: probs[i] = self.unigram[len(word)][letter] * 0.1
            except: probs[i] = 0
        best_idx = probs.index(max(probs))
        return self.letter_set[best_idx], max(probs)

ngram_engine = NgramEngine('Data/words_250000_train.txt')


In [ ]:
import secrets
pool = ['astri', 'innovation', 'technology', 'microelectronics', 'optimization', 'principal', 'artificial', 'intelligence', 'transformer', 'bidirectional', 'blockchain', 'cybersecurity', 'algorithm', 'architecture', 'semiconductor', 'cryptography', 'asynchronous', 'parallelism', 'orchestration', 'scalability', 'quantization', 'deployment', 'infrastructure', 'automation', 'benchmarking', 'probability', 'statistical', 'deeplearning', 'distributed', 'recursion']
random.shuffle(pool)

class OfficialAPIDemo:
    def __init__(self, target, game_id):
        self.target = target.lower()
        self.game_id = f'game_{game_id:04x}'
        self.guessed = []
        self.display = ['_'] * len(target)
        self.lives = 6

    def run(self):
        print(f"Successfully start a new game! Game ID: {self.game_id}. # of tries remaining: {self.lives}. Word: {' '.join(self.display)}.")
        while self.lives > 0 and '_' in self.display:
            ng_char, ng_conf = ngram_engine.guess(''.join(self.display), self.guessed)
            t_nn = encode_for_nn(''.join(self.display))
            with torch.no_grad():
                nn_probs = torch.softmax(model(t_nn), dim=1).squeeze()
                for idx in torch.argsort(nn_probs, descending=True):
                    nn_char = string.ascii_lowercase[idx.item()]
                    if nn_char not in self.guessed: break
            
            # Hybrid: Weighted confidence (bias towards Neural for OOD)
            guess = nn_char if nn_probs[idx].item() > (ng_conf * 1000) else ng_char
            print(f"Guessing letter: {guess}")
            self.guessed.append(guess)
            
            if guess in self.target:
                for i, c in enumerate(self.target): 
                    if c == guess: self.display[i] = guess
            else:
                self.lives -= 1
            
            res = {"game_id": self.game_id, "status": "ongoing", "tries_remains": self.lives, "word": " ".join(self.display)}
            print(f"Sever response: {res}")
            time.sleep(0.1)

        if '_' not in self.display:
            print(f"Successfully finished game: {self.game_id}\n")
            return True
        print(f"Failed game: {self.game_id}. Because of: # of tries exceeded!\n")
        return False

def start_session(limit=30):
    wins = 0
    for i, word in enumerate(pool[:limit]):
        demo = OfficialAPIDemo(word, secrets.randbits(16))
        if demo.run(): wins += 1
        print(f"[SESSION] Progress: {i+1}/{limit} | Wins: {wins} | Success Rate: {wins/(i+1):.1%}")
        print('=' * 60)
        time.sleep(0.5)
    print(f"\n🔥 ACCURACY: {wins/limit:.1%}")

start_session(30)